# Baselines and the new-customer path

**Owner:** Member 2 (Baselines & New Customers) · **Branch:** `m2-baselines` · **Code:** `src/hm_recsys/baselines.py`

## Questions this notebook answers
1. **What does today's random display cost?** We score random recommendations, the company's current strategy.
2. **How far do simple, non-personalised ideas get us?** We try all-time best sellers, "trending now" (last 1–2 weeks), best sellers by age band, and "buy again".
3. **What do we show a customer we know nothing about?** We build the *new-customer path*: trends for their age band, or global trends if we don't know their age.
4. **What is the catch?** Popular items take most of the exposure. We measure how concentrated the recommendations are and which products are never shown.
5. **Which baseline must the personalised models beat?** "Trending now", not random. The last section explains why.

## Checklist
- [x] Random (today's situation)
- [x] Global popularity
- [x] Recent popularity / "trending now" (1–2 weeks)
- [x] Popularity by age band (demographic filtering)
- [x] Buy again
- [x] Score all with `evaluate()`, by segment and by age band
- [x] Popularity bias: concentration and never-shown items
- [x] Why "trending now" is the real baseline to beat

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hm_recsys.data import RAW_DIR, assign_segments, load_split
from hm_recsys.evaluation import evaluate
from hm_recsys.baselines import (
    AgeBandPopularityRecommender, BuyAgainRecommender, NewCustomerRecommender,
    PopularityRecommender, RandomRecommender,
    customer_age_bands, exposure_table, lorenz_curve, popularity_bias_summary,
)

FIG_DIR = Path("../reports/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.float_format", "{:.4f}".format)

## 1. Data

We use the shared train/test split from Member 1 (`load_split`): the last 12 weeks for training and the week after for testing, split by time so no future information leaks into training.

Here we only need two small columns from the customer table (age) and a few product attributes for the bias analysis, so we read just those.

In [ ]:
train, test = load_split()

customers = pd.read_csv(RAW_DIR / "customers.csv", usecols=["customer_id", "age"])
articles = pd.read_csv(
    RAW_DIR / "articles.csv",
    usecols=["article_id", "prod_name", "product_type_name", "product_group_name",
             "index_group_name", "colour_group_name"],
    dtype={"article_id": "int64"},
)

test_customers = pd.Series(test["customer_id"].unique())
segments = assign_segments(train, test_customers)
age_bands = customer_age_bands(customers).reindex(test_customers).fillna("unknown")

print(f"train: {len(train):,} rows, {train['t_dat'].min():%Y-%m-%d} to {train['t_dat'].max():%Y-%m-%d}")
print(f"test : {len(test):,} rows, {test['customer_id'].nunique():,} customers")
print("test customers by segment:"); print(segments.value_counts())

In [ ]:
def tidy(results: pd.DataFrame, model_name: str) -> pd.DataFrame:
    """Put evaluate() output in long form: one row per (model, group)."""
    r = results.copy()
    if "segment" in r.columns:
        r = r.set_index("segment")
    r.index.name = "group"
    return r.reset_index().assign(model=model_name)

def col(df: pd.DataFrame, key: str) -> str:
    """Find a metric column by part of its name, e.g. col(df, 'hit') -> 'hit_rate@12'."""
    return next(c for c in df.columns if key.lower() in c.lower())

## 2. Choosing the "trending now" window (validation, not test)

How many recent weeks should "trending now" count: 1, 2 or 4? To choose without touching the test week, we hold out the **last week of the training data** as a validation week, fit on the weeks before it, and pick the window that scores best there. Then we refit on all training data. The same idea as the test split, just one week earlier.

In [ ]:
val_start = train["t_dat"].max() - pd.Timedelta(days=6)
fit_part, val_part = train[train["t_dat"] < val_start], train[train["t_dat"] >= val_start]

window_rows = []
for weeks in [1, 2, 4, None]:
    m = PopularityRecommender(weeks=weeks).fit(fit_part)
    window_rows.append(tidy(evaluate(m, val_part), m.name).query("group == 'all'"))
window_scores = pd.concat(window_rows, ignore_index=True).set_index("model")
display(window_scores)

best = window_scores[col(window_scores, "map")].idxmax()
TREND_WEEKS = None if best == "popularity" else int(best.split("_")[1].rstrip("w"))
print("chosen trending window (weeks):", TREND_WEEKS)

**Reading this:** fashion moves fast. If the shortest window wins, it means last week's best sellers predict next week far better than best sellers from three months ago, because collections, weather and campaigns change week to week. We use the winning window for every popularity-based model below.

## 3. The five baselines

| Model | What the customer sees | Personal data needed |
|---|---|---|
| **Random** | Random products from the catalogue (today) | none |
| **Global popularity** | All-time best sellers of the training period | none |
| **Trending now** | Best sellers of the last week(s) | none |
| **Popularity by age band** | Trending among people of the same age | age |
| **Buy again** | Products the customer bought before, topped up with age-band trends | purchase history |

Popularity counts **distinct buyers**, not rows, so one shopper buying ten of the same item does not make it "popular". Every model returns 12 products for *any* customer ID, including ones it has never seen.

In [ ]:
models = {
    "Random (today)": RandomRecommender(seed=42),
    "Global popularity": PopularityRecommender(weeks=None),
    "Trending now": PopularityRecommender(weeks=TREND_WEEKS),
    "Popularity by age band": AgeBandPopularityRecommender(weeks=TREND_WEEKS),
    "Buy again": BuyAgainRecommender(fallback=NewCustomerRecommender(weeks=TREND_WEEKS)),
}
for name, m in models.items():
    m.fit(train, articles, customers)

by_segment = pd.concat(
    [tidy(evaluate(m, test, segments=segments), name) for name, m in models.items()],
    ignore_index=True,
)
by_segment.set_index(["group", "model"]).sort_index()

In [ ]:
HIT, MAP = col(by_segment, "hit"), col(by_segment, "map")
order = list(models)
groups = [g for g in ["all", "new", "light", "regular"] if g in set(by_segment["group"])]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metric, title in [(axes[0], HIT, "Hit rate: share of customers who bought ≥1 recommended item"),
                          (axes[1], MAP, "MAP@12: how well the list is ranked")]:
    (by_segment.pivot(index="group", columns="model", values=metric)
               .loc[groups, order].plot.bar(ax=ax, rot=0, width=0.8))
    ax.set_title(title, fontsize=10); ax.set_xlabel("customer segment"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR / "m2_baselines_by_segment.png", dpi=150); plt.show()

summary = by_segment.query("group == 'all'").set_index("model").loc[order, [HIT, MAP]]
rand_hit = summary.loc["Random (today)", HIT]
summary["hit rate vs random"] = summary[HIT] / rand_hit if rand_hit > 0 else np.inf
summary

**What this means for the business** *(check these against the numbers above after running)*

- **Random display almost never shows a customer something they go on to buy.** With over 100,000 products in the catalogue, 12 random picks are a needle-in-a-haystack bet. This is the cost of today's strategy.
- **Simply showing what is selling right now multiplies hits many times over**, for every customer, with no personal data at all. It is the cheapest quick win available.
- **Age bands help where tastes differ by age** (compare the age-band chart below). It is a light form of personalisation that works even for first-time visitors, as long as we know their age.
- **"Buy again" is strongest for customers with history** (light and regular), because people often re-buy basics (check which product types dominate its hits). For new customers it falls back to age-band trends, so it never does worse than the new-customer path.

## 4. Results by age band

`evaluate()` breaks results down by any grouping we give it, so we pass the age bands instead of the purchase segments.

In [ ]:
by_age = pd.concat(
    [tidy(evaluate(models[n], test, segments=age_bands), n)
     for n in ["Random (today)", "Trending now", "Popularity by age band"]],
    ignore_index=True,
)
band_order = [b for b in ["16-24", "25-34", "35-44", "45-54", "55-64", "65+", "unknown"]
              if b in set(by_age["group"])]
ax = (by_age.pivot(index="group", columns="model", values=HIT)
            .loc[band_order].plot.bar(rot=0, figsize=(10, 4)))
ax.set_title("Hit rate by age band"); ax.set_xlabel("age band"); ax.set_ylabel("hit rate")
plt.tight_layout(); plt.savefig(FIG_DIR / "m2_hit_rate_by_age.png", dpi=150); plt.show()

In [ ]:
# What each age band is shown: the top 5 product types per band
age_model = models["Popularity by age band"]
top_types = {}
for band in band_order:
    ids = age_model.recommend_band(band, k=12)
    top_types[band] = (articles.set_index("article_id").loc[ids, "product_type_name"]
                       .value_counts().head(5).index.tolist())
pd.DataFrame.from_dict(top_types, orient="index", columns=[f"#{i}" for i in range(1, 6)])

**Reading this:** if the age-band model beats global trending mostly for the youngest and oldest bands, that is because those groups buy differently from the average customer, which is dominated by 25–45 year olds. Where the bars are equal, age adds nothing and global trends are enough.

## 5. The new-customer path

A **new** customer has no purchases in the training period, so collaborative filtering and "buy again" have nothing to work with. Our rule:

1. **Age known** → best sellers of the last week(s) among people in the same age band.
2. **Age unknown, or a customer ID we have never seen** → global "trending now".
3. If an age band has fewer than 12 trending items, top it up with global trends, so the list is always full.

This is `NewCustomerRecommender` in `baselines.py`, which Member 5's switching hybrid uses for the *new* segment.

In [ ]:
new_only = by_segment.query("group == 'new'").set_index("model").loc[
    ["Random (today)", "Global popularity", "Trending now", "Popularity by age band"]]
display(new_only[[HIT, MAP]])

new_path = NewCustomerRecommender(weeks=TREND_WEEKS).fit(train, articles, customers)
print("never-seen customer ->", new_path.recommend("this-id-does-not-exist"))

## 6. Popularity bias: who gets left out?

Popularity-based models show everyone almost the same few products. That is good for short-term hits but has costs:
- **Long-tail products are never shown**, so they can never become popular: a self-reinforcing loop.
- **New collections get no exposure** until they are already selling.
- Customers with less mainstream taste are served poorly.

We measure, for each model, how many different products anyone sees, what share of all recommendation slots go to the top 1% of best sellers, and how much of last quarter's sales came from products that are never recommended. (The Gini index from `evaluate()` gives the same picture as a single number.)

In [ ]:
preds = {name: m.recommend_all(test_customers) for name, m in models.items()}
bias = pd.DataFrame({name: popularity_bias_summary(p, train) for name, p in preds.items()}).T
gini_cols = [c for c in by_segment.columns if "gini" in c.lower() or "coverage" in c.lower()]
bias = bias.join(by_segment.query("group == 'all'").set_index("model")[gini_cols])
bias.loc[order]

In [ ]:
n_items = train["article_id"].nunique()
fig, ax = plt.subplots(figsize=(5.5, 5.5))
for name in order:
    x, y = lorenz_curve(preds[name], n_items)
    ax.plot(x, y, label=name)
ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="perfectly even")
ax.set_xlabel("share of products (least → most shown)"); ax.set_ylabel("share of recommendation slots")
ax.set_title("How concentrated are recommendations?"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR / "m2_lorenz_curves.png", dpi=150); plt.show()

In [ ]:
# Which kinds of products never get shown by "trending now"?
exp = exposure_table(preds["Trending now"], train, articles)
by_group = (exp.assign(never_shown=exp["shown"] == 0)
              .groupby("product_group_name")
              .agg(items_sold=("article_id", "size"), units_sold=("sold", "sum"),
                   share_items_never_shown=("never_shown", "mean"), slots=("shown", "sum")))
by_group["share_of_sales"] = by_group["units_sold"] / by_group["units_sold"].sum()
by_group["share_of_slots"] = by_group["slots"] / by_group["slots"].sum()
by_group.sort_values("units_sold", ascending=False)[
    ["items_sold", "share_of_sales", "share_of_slots", "share_items_never_shown"]].head(12)

**Reading this:** a product group with a large share of sales but a tiny share of recommendation slots (or none) is being ignored by popularity models, even though customers clearly want it. These gaps are what personalised models (Members 3 and 4) and the exploration idea in Member 5's bandit sketch are meant to close.

## 7. Why "trending now" is the real baseline to beat

Comparing our final system with random display would make any model look brilliant, because random is so weak. The honest question for the CEO is: **is personalisation worth it compared with the simplest sensible alternative?** That alternative is "trending now":

1. **It is nearly free.** One count of last week's sales, one list for everyone. No personal data, no model to maintain, easy to explain.
2. **It is already much stronger than random**, as section 3 shows, so most of the gain over today comes from *any* sensible strategy.
3. **It reacts to fashion cycles.** New collections, seasons and campaigns show up in it within a week.
4. **It is safe.** It needs no customer data, so it raises no privacy concerns, and it works for every customer from day one.

So in the business case we report **two numbers** for every model: the gain over random (what the company gains versus today) and the gain over trending now (what *personalisation itself* adds). A personalised model that cannot beat trending now is not worth its extra complexity.

In [ ]:
headline = by_segment.query("group == 'all'").set_index("model").loc[order, [HIT, MAP]]
trend_hit = headline.loc["Trending now", HIT]
for name, row in headline.iterrows():
    print(f"{name:<24} hit rate {row[HIT]:6.2%} | "
          f"x{row[HIT] / max(rand_hit, 1e-9):6.1f} vs random | x{row[HIT] / max(trend_hit, 1e-9):4.2f} vs trending")

# hand-off for Member 5 (business value) and the main pipeline
out = Path("../reports"); out.mkdir(exist_ok=True)
by_segment.to_csv(out / "m2_baseline_results.csv", index=False)
print("saved reports/m2_baseline_results.csv")

## 8. Quick wins and hand-off

**Quick wins the company could ship next week, with no machine learning:**
1. Replace random display with **"trending now"**: one list for everyone, refreshed weekly.
2. Where age is known, show **trending in your age band** instead.
3. For returning customers, add a **"buy it again"** row for basics.

**For the team:**
- `PopularityRecommender(weeks=...)` is the fallback for Members 3 and 4 when a customer or item is unknown to their models.
- `NewCustomerRecommender` is the *new* segment path in Member 5's `SwitchingHybrid`.
- Results for the business value model are saved in `reports/m2_baseline_results.csv`; figures are in `reports/figures/m2_*.png`.